# Hybrid Recommender (Stage 1: CF + Popularity)

The item-based CF model produced better ranking metrics (MRR, nDCG) than the popularity baseline, but a slightly worse AUC. The most likely cause: CF assigns score 0 to cold-start articles it has never seen, which hurts discrimination on the full candidate list.

A hybrid fixes this by combining multiple signals. Stage 1 blends **collaborative filtering** with the **popularity baseline**. When Stage 2 comes (content-based filtering from my groupmate), we will plug it into the same scoring function as a third signal — no restructuring needed.

**Blending strategy:** for each impression, min-max normalise each signal across its candidates (so every signal is on a 0–1 scale), then take a weighted sum.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize
from sklearn.metrics import roc_auc_score

TRAIN_BEHAVIORS = "../smallDataset/MINDsmall_train/behaviors.tsv"
DEV_BEHAVIORS   = "../smallDataset/MINDsmall_dev/behaviors.tsv"

COLS = ["impression_id", "user_id", "time", "history", "impressions"]

## Step 1 — Build popularity scores (from the baseline model)

Count how many times each article was clicked in training. Same as the baseline notebook.

In [ ]:
train = pd.read_csv(TRAIN_BEHAVIORS, sep="\t", header=None, names=COLS)

click_counts = Counter()
user_clicks  = defaultdict(set)

for _, row in train.iterrows():
    uid = row["user_id"]

    # History clicks (used for CF matrix)
    if isinstance(row["history"], str) and row["history"].strip():
        for nid in row["history"].split():
            user_clicks[uid].add(nid)

    # Impression clicks (used for both popularity and CF)
    if isinstance(row["impressions"], str):
        for item in row["impressions"].split():
            nid, label = item.rsplit("-", 1)
            if label == "1":
                click_counts[nid] += 1
                user_clicks[uid].add(nid)

print(f"Popularity: {len(click_counts)} articles, {sum(click_counts.values())} total clicks")
print(f"CF matrix: {len(user_clicks)} users, {sum(len(v) for v in user_clicks.values())} interactions")

## Step 2 — Build the item-based CF interaction matrix

Same sparse matrix construction as the CF notebook: rows = users, columns = articles, then transpose + L2 normalise so we can compute cosine similarity on demand.

In [ ]:
user_to_idx    = {u: i for i, u in enumerate(user_clicks.keys())}
all_articles   = sorted({nid for clicks in user_clicks.values() for nid in clicks})
article_to_idx = {a: i for i, a in enumerate(all_articles)}

rows, cols = [], []
for uid, clicks in user_clicks.items():
    u_idx = user_to_idx[uid]
    for nid in clicks:
        rows.append(u_idx)
        cols.append(article_to_idx[nid])

data = np.ones(len(rows), dtype=np.float32)
interaction = csr_matrix((data, (rows, cols)), shape=(len(user_to_idx), len(article_to_idx)))

article_user_norm = normalize(interaction.T.tocsr(), norm="l2", axis=1, copy=False)

print(f"Interaction matrix: {interaction.shape}, nnz={interaction.nnz}")

## Step 3 — Define the individual scoring functions

Each signal is its own `score_fn` with the same interface `(history, candidates) -> list[float]`. Keeping them separate makes it trivial to add a third signal (CBF) later — we just write another function with the same shape.

In [ ]:
def popularity_score(history, candidates):
    """Return training click count for each candidate (0 if unseen)."""
    return [float(click_counts.get(nid, 0)) for nid in candidates]


def cf_score(history, candidates):
    """Average cosine similarity of each candidate to the user's history items."""
    hist_idx = [article_to_idx[h] for h in history if h in article_to_idx]
    if not hist_idx:
        return [0.0] * len(candidates)

    hist_vecs = article_user_norm[hist_idx]  # (h, n_users)

    scores = []
    for cand in candidates:
        c_idx = article_to_idx.get(cand)
        if c_idx is None:
            scores.append(0.0)
            continue
        cand_vec = article_user_norm[c_idx]
        sims = np.asarray((cand_vec @ hist_vecs.T).todense()).ravel()
        scores.append(float(sims.mean()))
    return scores


# Placeholder for Stage 2 — plug in CBF from the groupmate here
def cbf_score(history, candidates):
    """STUB — will be replaced with content-based filtering in Stage 2."""
    return [0.0] * len(candidates)

## Step 4 — Normalisation helper

Different signals live on different scales:
- Popularity scores can be in the thousands (raw click counts)
- CF scores are bounded cosine similarities (0–1)
- CBF scores will have their own scale

Before blending, we **min-max normalise** each signal across the candidates of a single impression. This puts every signal on a 0–1 scale so weights are meaningful and no signal dominates purely because of its raw magnitude.

In [ ]:
def minmax(scores):
    """Min-max normalise a score list to the [0, 1] range."""
    arr = np.asarray(scores, dtype=np.float32)
    lo, hi = arr.min(), arr.max()
    if hi - lo < 1e-12:
        return np.zeros_like(arr)
    return (arr - lo) / (hi - lo)

## Step 5 — The hybrid scoring function

Configurable weights per signal. Stage 1 uses only CF and popularity. Stage 2 will turn on the CBF weight.

In [ ]:
# Stage 1 weights — CBF weight is 0 until the groupmate's model lands
WEIGHTS = {
    "cf":  0.7,
    "pop": 0.3,
    "cbf": 0.0,
}

def hybrid_score(history, candidates):
    cf_raw  = cf_score(history, candidates)
    pop_raw = popularity_score(history, candidates)
    cbf_raw = cbf_score(history, candidates)

    cf_n  = minmax(cf_raw)
    pop_n = minmax(pop_raw)
    cbf_n = minmax(cbf_raw)

    blended = (
        WEIGHTS["cf"]  * cf_n +
        WEIGHTS["pop"] * pop_n +
        WEIGHTS["cbf"] * cbf_n
    )
    return blended.tolist()

## Step 6 — Evaluation

Same evaluation loop as the other notebooks so results are directly comparable.

In [ ]:
def dcg_at_k(relevance, k):
    relevance = np.array(relevance[:k], dtype=np.float32)
    if relevance.size == 0:
        return 0.0
    return float((relevance / np.log2(np.arange(2, relevance.size + 2))).sum())

def ndcg_at_k(relevance, k):
    ideal = sorted(relevance, reverse=True)
    idcg = dcg_at_k(ideal, k)
    return dcg_at_k(relevance, k) / idcg if idcg > 0 else 0.0

def mrr(relevance):
    for i, r in enumerate(relevance):
        if r == 1:
            return 1.0 / (i + 1)
    return 0.0

def evaluate(behaviors_path, score_fn):
    aucs, mrrs, ndcg5s, ndcg10s = [], [], [], []
    skipped = 0

    with open(behaviors_path, encoding="utf-8") as f:
        for line in f:
            cols = line.strip().split("\t")
            if len(cols) < 5:
                continue

            history    = cols[3].strip().split() if cols[3].strip() else []
            impressions = cols[4].strip().split()

            candidates, labels = [], []
            for imp in impressions:
                parts = imp.rsplit("-", 1)
                if len(parts) != 2:
                    continue
                nid, label = parts
                candidates.append(nid)
                labels.append(int(label))

            if sum(labels) == 0 or sum(labels) == len(labels):
                skipped += 1
                continue

            scores = score_fn(history, candidates)
            ranked = [lbl for _, lbl in sorted(zip(scores, labels), reverse=True)]

            aucs.append(roc_auc_score(labels, scores))
            mrrs.append(mrr(ranked))
            ndcg5s.append(ndcg_at_k(ranked, 5))
            ndcg10s.append(ndcg_at_k(ranked, 10))

    print(f"Evaluated {len(aucs)} impressions | Skipped (degenerate): {skipped}")
    return {
        "AUC":     round(float(np.mean(aucs)), 4),
        "MRR":     round(float(np.mean(mrrs)), 4),
        "nDCG@5":  round(float(np.mean(ndcg5s)), 4),
        "nDCG@10": round(float(np.mean(ndcg10s)), 4),
    }

In [ ]:
metrics = evaluate(DEV_BEHAVIORS, hybrid_score)

print("\nHybrid (CF + Popularity) — Dev Set Results")
print("-" * 45)
for k, v in metrics.items():
    print(f"  {k:<10} {v}")

## Step 7 — Comparison

| Model | AUC | MRR | nDCG@5 | nDCG@10 |
|---|---|---|---|---|
| Baseline (popularity) | 0.5318 | 0.3387 | 0.3321 | 0.3982 |
| Item-based CF         | 0.5209 | 0.3671 | 0.3679 | 0.4308 |
| **Hybrid (Stage 1)**  | *fill in* | *fill in* | *fill in* | *fill in* |

**What we hope to see:** the hybrid should keep the ranking-metric gains from CF while recovering the AUC we lost. If it does, that confirms the cold-start hypothesis from the CF notebook — popularity is filling in where CF has no signal.

**Tuning:** the current weights (CF 0.7, Pop 0.3) are a reasonable starting guess. Try a few other splits (0.5/0.5, 0.8/0.2) to see how sensitive the metrics are. Include a short weight-sweep table in the report if time allows.

## Step 8 — How to add CBF later (Stage 2)

When the content-based filtering model is ready from the groupmate, two small changes are all that is needed:

1. Replace the body of `cbf_score(history, candidates)` with the real CBF scoring logic
2. Update the `WEIGHTS` dictionary to include a non-zero `cbf` weight (e.g. `cf: 0.5, pop: 0.2, cbf: 0.3`)

No other code changes. The `hybrid_score` function already min-max normalises each signal and blends them, so a new signal slots in cleanly.

## Step 9 — Reflection

Why this hybrid structure is reasonable:
- **Addresses a real observed weakness** — the cold-start AUC drop in pure CF motivates the popularity fallback
- **Interpretable** — each signal is a separate function with a clear meaning; you can turn them on and off via the weights
- **Extensible** — adding CBF requires no restructuring, only a new function and a weight
- **Per-impression normalisation** — putting every signal on a 0–1 scale means weights directly reflect the relative importance of each signal

Limitations:
- Weights are hand-picked, not learned
- Min-max normalisation is sensitive to outliers within an impression
- Linear blending cannot capture "use CF when history is rich, popularity when history is empty" — that would require a maturity-aware blending function